# LightGBM — r/Random_Acts_Of_Pizza request fulfillment

Binary classification, AUC-optimized. Hyperparameters from `CLAUDE.md` model notes.
Goal: beat the Logistic Regression baseline of **AUC 0.6452**.

Pipeline mirrors `xgboost.ipynb` so the two models can be compared on byte-identical splits and OOF predictions.

In [1]:
# Setup: on Colab, clone the repo so utils.py is importable and install needed packages.
# No-op when running locally with the conda env.
import os, sys, subprocess

REPO_URL = "https://github.com/JaeHub/the-free-pizza-project"
REPO_BRANCH = "GB"
REPO_DIR = "/content/the-free-pizza-project"

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, REPO_DIR], check=True)
    else:
        subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)
    os.chdir(REPO_DIR)
    subprocess.run(["pip", "install", "-q", "kagglehub", "xgboost", "lightgbm"], check=True)

print("in_colab:", IN_COLAB, "| cwd:", os.getcwd())

in_colab: True | cwd: /content/the-free-pizza-project


In [2]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import roc_auc_score

from utils import (
    TARGET_COLUMN,
    RANDOM_STATE,
    load_data,
    engineer_features,
    get_train_test_split,
    get_cv_folds,
    save_predictions,
    evaluate_cv,
    evaluate_test,
    plot_feature_importance,
    log_run,
    describe_target,
    plot_cv_fold_aucs,
    plot_cv_learning_curves,
    inspect_predictions,
    print_summary,
)

MODEL_NAME = "lightgbm"
print("lightgbm version:", lgb.__version__)

ImportError: cannot import name 'describe_target' from 'utils' (/content/the-free-pizza-project/utils.py)

## Load & engineer features

In [ ]:
df = load_data()
X = engineer_features(df)
y = df[TARGET_COLUMN].astype(int).values

print(f"Shape: {df.shape}\n")
describe_target(y, df)

text_len = (df["request_title"].fillna("") + " " + df["request_text_edit_aware"].fillna("")).str.split().str.len()
print(f"\nAvg request length : {text_len.mean():.0f} words")
print(f"Engineered features: {X.shape[1]}  →  {list(X.columns)}")
X.head()

## Train/test split (shared across all models)

In [ ]:
train_idx, test_idx = get_train_test_split(y)
X_train, X_test = X.iloc[train_idx].reset_index(drop=True), X.iloc[test_idx].reset_index(drop=True)
y_train, y_test = y[train_idx], y[test_idx]

print(f"Train: {len(y_train)} | Test: {len(y_test)}  (5-fold CV on train; no separate val split)")
print(f"  train pos rate: {y_train.mean():.3f}  |  test pos rate: {y_test.mean():.3f}")

## 5-fold stratified CV with early stopping

Per CLAUDE.md, use `is_unbalance=True` (not both `is_unbalance` and `scale_pos_weight`).

In [ ]:
BASE_PARAMS = dict(
    objective="binary",
    metric="auc",
    num_leaves=31,
    learning_rate=0.05,
    n_estimators=1000,
    min_child_samples=20,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    is_unbalance=True,
    random_state=RANDOM_STATE,
    verbosity=-1,
)

folds = get_cv_folds(y_train)
oof_probs = np.zeros(len(y_train), dtype=float)
best_iters = []
evals_results = []

for fold_i, (tr_idx, val_idx) in enumerate(folds):
    X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train[tr_idx], y_train[val_idx]

    model = lgb.LGBMClassifier(**BASE_PARAMS)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(50, verbose=False)],
    )

    evals_results.append(model.evals_result_)
    oof_probs[val_idx] = model.predict_proba(X_val)[:, 1]
    best_iters.append(model.best_iteration_)
    fold_auc = roc_auc_score(y_val, oof_probs[val_idx])
    print(f"fold {fold_i}: best_iter={model.best_iteration_:4d}  val AUC={fold_auc:.4f}")

print("median best_iteration:", int(np.median(best_iters)))

In [ ]:
cv_results = evaluate_cv(y_train, oof_probs, MODEL_NAME)

[lightgbm] CV AUC: 0.6352 ± 0.0223  (per-fold: [np.float64(0.6191), np.float64(0.6225), np.float64(0.644), np.float64(0.675), np.float64(0.6152)], overall OOF: 0.6183)


In [ ]:
plot_cv_fold_aucs(cv_results["fold_aucs"], cv_results["mean"], MODEL_NAME)
plot_cv_learning_curves(evals_results, MODEL_NAME)

## Refit on full train, evaluate once on held-out test

In [ ]:
n_folds = len(folds)
final_n_estimators = int(np.median(best_iters) * n_folds / (n_folds - 1))

FINAL_PARAMS = {**BASE_PARAMS, "n_estimators": final_n_estimators}
final_model = lgb.LGBMClassifier(**FINAL_PARAMS)
final_model.fit(X_train, y_train)

train_probs = final_model.predict_proba(X_train)[:, 1]
test_probs = final_model.predict_proba(X_test)[:, 1]
refit_train_auc = roc_auc_score(y_train, train_probs)

test_metrics = evaluate_test(y_test, test_probs, MODEL_NAME)

## Save artifacts and inspect feature importance

In [ ]:
save_predictions(MODEL_NAME, oof_probs, test_probs)
importances = final_model.booster_.feature_importance(importance_type="gain")
plot_feature_importance(importances, X_train.columns, MODEL_NAME)
log_run(MODEL_NAME, FINAL_PARAMS, cv_results["mean"], cv_results["std"], test_metrics["test_auc"])

## Inspect predictions

Highest-confidence correct calls vs. confident wrong calls — same diagnostic table as `cnn_model.ipynb`.

In [ ]:
y_pred = (test_probs >= 0.5).astype(int)
test_titles = df.iloc[test_idx]["request_title"].reset_index(drop=True)
inspect_predictions(test_titles, y_test, y_pred, test_probs, MODEL_NAME, n=3)

## Key Findings Summary

In [ ]:
top_features = (
    pd.Series(importances, index=X_train.columns)
    .sort_values(ascending=False)
    .head(3)
    .index.tolist()
)
overfit_gap = refit_train_auc - cv_results["mean"]
overfit_note = "modest overfit" if overfit_gap < 0.05 else "noticeable overfit (consider lowering num_leaves or raising min_child_samples)"
beats_lr = test_metrics["test_auc"] > 0.6452
lr_note = (
    f"beats LR baseline (Δ = {test_metrics['test_auc']-0.6452:+.4f})"
    if beats_lr else f"below LR baseline (Δ = {test_metrics['test_auc']-0.6452:+.4f})"
)

print_summary(
    MODEL_NAME,
    dataset={
        "total_samples": len(df),
        "fulfillment_rate_pct": float(df[TARGET_COLUMN].mean() * 100),
        "n_features": X.shape[1],
    },
    model={
        "name": "LightGBM",
        "params_summary": (
            f"num_leaves={FINAL_PARAMS['num_leaves']}, lr={FINAL_PARAMS['learning_rate']}, "
            f"n_estimators={FINAL_PARAMS['n_estimators']}, min_child_samples={FINAL_PARAMS['min_child_samples']}, "
            f"subsample={FINAL_PARAMS['subsample']}, colsample={FINAL_PARAMS['colsample_bytree']}"
        ),
        "class_handling": "is_unbalance=True",
    },
    training={
        "cv_auc_mean": cv_results["mean"],
        "cv_auc_std": cv_results["std"],
        "fold_aucs": cv_results["fold_aucs"],
        "median_best_iter": int(np.median(best_iters)),
        "refit_train_auc": refit_train_auc,
    },
    test={
        "roc_auc": test_metrics["test_auc"],
        "f1_fulfilled": test_metrics["f1_fulfilled"],
        "accuracy_pct": test_metrics["accuracy"] * 100,
    },
    takeaways=[
        f"Test AUC {lr_note}; CNN baseline 0.5846.",
        f"Top 3 features by importance (gain): {', '.join(top_features)}.",
        f"Refit-train vs CV AUC gap = {overfit_gap:+.4f} → {overfit_note}.",
        f"Early stopping picked median {int(np.median(best_iters))} trees of {BASE_PARAMS['n_estimators']} budget — well below the cap.",
    ],
)